# ✍️ InkMind — TinyGPT Poem Generator

A small GPT-style Transformer trained on poetry to generate
poem continuations from user-provided prompts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
import math

import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
GPU available: False


## 2. Load the Poetry Dataset

I use the Poetry Foundation dataset as the training data for InkMind.

The `Poem` column contains the actual poetry text that the model will learn
from. The title is kept for reference, while other metadata is not needed
for language generation.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving PoetryFoundationData.csv to PoetryFoundationData.csv


In [ ]:
df = pd.read_csv("PoetryFoundationData.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (13854, 5)


,Unnamed: 0,Title,Poem,Poet,Tags
0,0,\r\r\n Objects Used to Prop...,"\r\r\nDog bone, stapler,\r\r\ncribbage board, ...",Michelle Menting,NaN
1,1,\r\r\n The New Church\r\r\n...,"\r\r\nThe old cupola glinted above the clouds,...",Lucia Cherciu,NaN
2,2,\r\r\n Look for Me\r\r\n ...,\r\r\nLook for me under the hood\r\r\nof that ...,Ted Kooser,NaN
3,3,\r\r\n Wild Life\r\r\n ...,"\r\r\nBehind the silo, the Mother Rabbit\r\r\n...",Grace Cavalieri,NaN
4,4,\r\r\n Umbrella\r\r\n ...,\r\r\nWhen I push your button\r\r\nyou fly off...,Connie Wanek,NaN


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Columns:
['Unnamed: 0', 'Title', 'Poem', 'Poet', 'Tags']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13854 entries, 0 to 13853
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  13854 non-null  int64 
 1   Title       13854 non-null  object
 2   Poem        13854 non-null  object
 3   Poet        13854 non-null  object
 4   Tags        12899 non-null  object
dtypes: int64(1), object(4)
memory usage: 541.3+ KB


## 3. Dataset Inspection

Before training, I check for missing values and duplicate poems.

Duplicate poems can cause the same training example to appear more than
once, so removing them helps avoid unnecessary repetition in the dataset.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate poems:", df["Poem"].duplicated().sum())

Missing values:
Unnamed: 0      0
Title           0
Poem            0
Poet            0
Tags          955
dtype: int64

Duplicate rows: 2
Duplicate poems: 100


In [ ]:
df = df[["Title", "Poem"]]

print("Selected columns:", df.columns.tolist())
print("Dataset shape:", df.shape)

Selected columns: ['Title', 'Poem']
Dataset shape: (13854, 2)


In [ ]:
df = df.drop_duplicates(subset="Poem").reset_index(drop=True)

print("Dataset shape after removing duplicate poems:", df.shape)
print("Duplicate poems:", df["Poem"].duplicated().sum())

Dataset shape after removing duplicate poems: (13754, 2)
Duplicate poems: 0


## 4. Text Cleaning

The original dataset contains different newline representations and some
Unicode whitespace or invisible characters.

I normalize these characters while preserving meaningful poetry formatting,
especially line breaks and punctuation.

In [ ]:
df["Poem"] = (
    df["Poem"]
    .str.replace(r"\r\n", "\n", regex=False)
    .str.replace(r"\r", "\n", regex=False)
    .str.strip()
)

print(df.iloc[0]["Poem"])




















a frame of air.


In [ ]:
space_chars = [
    "\u2002",  # EN SPACE
    "\u2003",  # EM SPACE
    "\u2006",  # SIX-PER-EM SPACE
    "\u2009",  # THIN SPACE
    "\u200a"   # HAIR SPACE
]

for char in space_chars:
    df["Poem"] = df["Poem"].str.replace(char, " ", regex=False)

df["Poem"] = df["Poem"].str.replace("\u2028", "\n", regex=False)
df["Poem"] = df["Poem"].str.replace("\r", "\n", regex=False)

remove_chars = [
    "\x9f",
    "\xad",
    "\u200b",
    "\u2060",
    "\ufeff"
]

for char in remove_chars:
    df["Poem"] = df["Poem"].str.replace(char, "", regex=False)

df["Poem"] = df["Poem"].str.strip()

In [ ]:
df = df[df["Poem"].str.strip() != ""].reset_index(drop=True)

df["poem_length"] = df["Poem"].str.len()

print("Number of poems:", len(df))
print("\nPoem length statistics:")
print(df["poem_length"].describe())

Number of poems: 13745

Poem length statistics:
count    13745.000000
mean      1508.934522
std       2610.466718
min          1.000000
25%        588.000000
50%        919.000000
75%       1592.000000
max      88693.000000
Name: poem_length, dtype: float64


### Checking Poem Lengths

I inspect the poem lengths to understand the range of text in the dataset.

I keep short poems rather than removing them arbitrarily. During training,
the poems will be divided into smaller fixed-length sequences.

In [ ]:
print("Shortest poem:")
print(df.loc[df["poem_length"].idxmin(), "Poem"])

print("\nLongest poem length:", df["poem_length"].max())
print("Empty poems:", (df["Poem"] == "").sum())
print("Poems under 50 characters:", (df["poem_length"] < 50).sum())

Shortest poem:
i

Longest poem length: 88693
Empty poems: 0
Poems under 50 characters: 56


## 5. Build the Poetry Corpus

I combine all poems into one text corpus.

`<|ENDOFPOEM|>` separates one poem from the next so the model can learn
where individual poems end.

In [ ]:
poems = df["Poem"].tolist()

corpus = "\n<|ENDOFPOEM|>\n".join(poems)

print("Number of poems:", len(poems))
print("Total characters:", len(corpus))

print("\nFirst 1000 characters:")
print(corpus[:1000])

Number of poems: 13745
Total characters: 20946465

First 1000 characters:
Dog bone, stapler,


cribbage board, garlic press


     because this window is loose—lacks


suction, lacks grip.


Bungee cord, bootstrap,


dog leash, leather belt


     because this window had sash cords.


They frayed. They broke.


Feather duster, thatch of straw, empty


bottle of Elmer's glue


     because this window is loud—its hinges clack


open, clack shut.


Stuffed bear, baby blanket,


single crib newel


     because this window is split. It's dividing


in two.


Velvet moss, sagebrush,


willow branch, robin's wing


     because this window, it's pane-less. It's only


a frame of air.
<|ENDOFPOEM|>
The old cupola glinted above the clouds, shone


among fir trees, but it took him an hour


for the half mile all the way up the hill. As he trailed,


the village passed him by, greeted him,


asked about his health, but everybody hurried


to catch the mass, left him leaning against fences,


me

## 6. Character-Level Tokenization

TinyGPT uses character-level tokenization.

Instead of treating a whole word as one token, each character becomes a
token. This makes the tokenizer simple and allows us to build the model
from scratch.

For example:

"Love"

becomes:

"L" → "o" → "v" → "e"

In [ ]:
chars = sorted(list(set(corpus)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print("Vocabulary size:", vocab_size)

Vocabulary size: 288


### Encoding and Decoding Test

`stoi` converts characters into integer IDs.

`itos` converts those integer IDs back into characters.

This test confirms that the tokenizer can encode and decode text correctly.

In [ ]:
sample = "Love is"

encoded = [stoi[ch] for ch in sample]
decoded = "".join(itos[i] for i in encoded)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: Love is
Encoded: [46, 81, 88, 71, 2, 75, 85]
Decoded: Love is


## 7. Convert the Corpus to Token IDs

Neural networks work with numbers rather than raw text.

I therefore convert every character in the corpus into its corresponding
integer token ID and store the result as a PyTorch tensor.

In [ ]:
data = torch.tensor(
    [stoi[ch] for ch in corpus],
    dtype=torch.long
)

print("Data shape:", data.shape)
print("Data type:", data.dtype)

Data shape: torch.Size([20946465])
Data type: torch.int64


In [ ]:
print("First 100 token IDs:")
print(data[:100])

print("\nDecoded:")
print("".join(itos[i.item()] for i in data[:100]))

First 100 token IDs:
tensor([ 38,  81,  73,   2,  68,  81,  80,  71,  14,   2,  85,  86,  67,  82,
         78,  71,  84,  14,   1,   1,   1,  69,  84,  75,  68,  68,  67,  73,
         71,   2,  68,  81,  67,  84,  70,  14,   2,  73,  67,  84,  78,  75,
         69,   2,  82,  84,  71,  85,  85,   1,   1,   1,   2,   2,   2,   2,
          2,  68,  71,  69,  67,  87,  85,  71,   2,  86,  74,  75,  85,   2,
         89,  75,  80,  70,  81,  89,   2,  75,  85,   2,  78,  81,  81,  85,
         71, 234,  78,  67,  69,  77,  85,   1,   1,   1,  85,  87,  69,  86,
         75,  81])

Decoded:
Dog bone, stapler,


cribbage board, garlic press


     because this window is loose—lacks


suctio


## 8. Training and Validation Data

I split the corpus into 90% training data and 10% validation data.

The training portion is used to update the model's parameters.

The validation portion is kept separate and is used to check how well the
model performs on text it did not use for training.

In [ ]:
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 18851818
Validation tokens: 2094647


## 9. Create Training Sequences

The model does not process the entire corpus at once.

`block_size` defines how many previous characters the model can see at a
time.

`batch_size` defines how many sequences are processed together.

For each input sequence `X`, the target `Y` is shifted by one character.
This creates the next-token prediction task.

In [ ]:
batch_size = 32
block_size = 256

In [ ]:
def get_batch(split):
    data_source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data_source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data_source[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x, y

In [ ]:
X, Y = get_batch("train")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([32, 256])
Y shape: torch.Size([32, 256])


In [ ]:
print("Input:")
print("".join(itos[i.item()] for i in X[0]))

print("\nTarget:")
print("".join(itos[i.item()] for i in Y[0]))

Input:
m-locked strutting


up the main hall of Central High unopposed


for decades looked. I flattened myself against


the wall, unnerved by their cloudsea of élan,


which pounced upon any timid girl regrettably


in their way, their high-wattage lifting slow

Target:
-locked strutting


up the main hall of Central High unopposed


for decades looked. I flattened myself against


the wall, unnerved by their cloudsea of élan,


which pounced upon any timid girl regrettably


in their way, their high-wattage lifting slow 


## 10. Token Embeddings

An embedding converts each token ID into a learned numerical vector.

Here, every character is represented by a vector of 128 numbers.

This gives the model a richer numerical representation than a simple
integer ID.

In [ ]:
n_embd = 128

token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
)

token_embeddings = token_embedding_table(X)

print("Token embedding shape:", token_embeddings.shape)

Token embedding shape: torch.Size([32, 256, 128])


## 11. Position Embeddings

Self-attention does not automatically know the order of tokens.

Position embeddings provide information about where each character occurs
in the sequence.

The token and position embeddings are then added together.

In [ ]:
position_embedding_table = nn.Embedding(
    block_size,
    n_embd
)

positions = torch.arange(block_size)

position_embeddings = position_embedding_table(positions)

print("Position embedding shape:", position_embeddings.shape)

Position embedding shape: torch.Size([256, 128])


In [ ]:
x = token_embeddings + position_embeddings

print("Combined embedding shape:", x.shape)

Combined embedding shape: torch.Size([32, 256, 128])


## 12. Query, Key and Value

Self-attention uses three learned representations:

- Query: what a token is looking for
- Key: what information a token offers for matching
- Value: the information that is actually passed forward

These are produced using learned linear layers.

In [ ]:
query_layer = nn.Linear(n_embd, n_embd)
key_layer = nn.Linear(n_embd, n_embd)
value_layer = nn.Linear(n_embd, n_embd)

Q = query_layer(x)
K = key_layer(x)
V = value_layer(x)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([32, 256, 128])
K shape: torch.Size([32, 256, 128])
V shape: torch.Size([32, 256, 128])


## 13. Attention Scores

The query and key vectors are compared using a dot product.

This produces a score showing how strongly each position relates to every
other position.

In [ ]:
attention_scores = Q @ K.transpose(-2, -1)

print("Attention scores shape:", attention_scores.shape)

Attention scores shape: torch.Size([32, 256, 256])


## 14. Scale the Attention Scores

The attention scores are divided by the square root of the key dimension.

Scaling prevents the values from becoming too large before applying
softmax, which helps make training more stable.

In [ ]:
d_k = K.size(-1)

scaled_scores = attention_scores / math.sqrt(d_k)

print("Scaled attention scores shape:", scaled_scores.shape)

Scaled attention scores shape: torch.Size([32, 256, 256])


## 15. Causal Mask

InkMind is an autoregressive language model.

When predicting the next character, it must not look at future characters.

The causal mask therefore allows each position to attend only to itself
and earlier positions.

In [ ]:
mask = torch.tril(
    torch.ones(block_size, block_size)
)

print("Mask shape:", mask.shape)
print(mask[:5, :5])

Mask shape: torch.Size([256, 256])
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])


In [ ]:
scaled_scores = scaled_scores.masked_fill(
    mask == 0,
    float("-inf")
)

print("Masked scores shape:", scaled_scores.shape)

Masked scores shape: torch.Size([32, 256, 256])


## 16. Attention Weights

Softmax converts the attention scores into probabilities.

The values for each position sum to 1, allowing the model to determine how
much attention to give to each previous position.

In [ ]:
attention_weights = torch.softmax(
    scaled_scores,
    dim=-1
)

print("Attention weights shape:", attention_weights.shape)

Attention weights shape: torch.Size([32, 256, 256])


In [ ]:
print("Example attention weights:")
print(attention_weights[0, 4, :5])

print("\nSum of weights:")
print(attention_weights[0, 4].sum())

Example attention weights:
tensor([0.0773, 0.3019, 0.3335, 0.1424, 0.1449], grad_fn=<SliceBackward0>)

Sum of weights:
tensor(1., grad_fn=<SumBackward0>)


## 17. Context Representation

The attention weights are multiplied by the value vectors.

This is where self-attention actually gathers information from other
positions and produces a context-aware representation for each token.

In [ ]:
context = attention_weights @ V

print("Context shape:", context.shape)

Context shape: torch.Size([32, 256, 128])


## 18. Layer Normalization

Before sending the contextual representations into the Feed-Forward
Network, I apply LayerNorm.

Normalization helps keep the values in a stable range during training.

In [ ]:
layer_norm = nn.LayerNorm(n_embd)

normalized_context = layer_norm(context)

print("Context shape:", context.shape)
print("Normalized context shape:", normalized_context.shape)

Context shape: torch.Size([32, 256, 128])
Normalized context shape: torch.Size([32, 256, 128])


## 19. Feed-Forward Network

The Feed-Forward Network processes each token's contextual representation
independently.

It first expands the representation from 128 to 512 dimensions, applies
GELU for non-linearity, and then projects it back to 128 dimensions.

Self-attention gathers information; the FFN transforms that gathered
information.

In [ ]:
ffn = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.GELU(),
    nn.Linear(4 * n_embd, n_embd)
)

print(ffn)

Sequential(
  (0): Linear(in_features=128, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Linear(in_features=512, out_features=128, bias=True)
)


In [ ]:
ffn_output = ffn(normalized_context)

print("FFN output shape:", ffn_output.shape)

FFN output shape: torch.Size([32, 256, 128])


## 20. Residual Connection

The residual connection adds the original contextual representation back
to the FFN output.

This helps preserve information from the earlier representation while
allowing the FFN to learn useful transformations.

In [ ]:
residual_output = context + ffn_output

print("Residual output shape:", residual_output.shape)

Residual output shape: torch.Size([32, 256, 128])


## 21. Predict the Next Character

The final linear layer converts the 128-dimensional representation into
one score for every character in the vocabulary.

These scores are called logits.

For our vocabulary, each position produces 288 possible character scores.

In [ ]:
lm_head = nn.Linear(
    n_embd,
    vocab_size
)

logits = lm_head(residual_output)

print("Logits shape:", logits.shape)

Logits shape: torch.Size([32, 256, 288])


## 22. Cross-Entropy Loss

The model predicts one character at each position.

Cross-entropy loss measures how different the model's predictions are from
the actual target characters.

A lower loss generally means the model is becoming better at next-character
prediction.

In [ ]:
B, T, C = logits.shape

logits_flat = logits.view(B * T, C)
targets_flat = Y.view(B * T)

print("Logits shape:", logits_flat.shape)
print("Targets shape:", targets_flat.shape)

Logits shape: torch.Size([8192, 288])
Targets shape: torch.Size([8192])


In [ ]:
loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(
    logits_flat,
    targets_flat
)

print("Initial loss:", loss.item())

Initial loss: 5.667099952697754


## 23. Prepare the Model for Training

Because the model components were built as separate layers, I place them
inside an `nn.ModuleList`.

This allows PyTorch to track all trainable parameters and lets the optimizer
update them during training.

In [ ]:
model_parts = nn.ModuleList([
    token_embedding_table,
    position_embedding_table,
    query_layer,
    key_layer,
    value_layer,
    ffn,
    layer_norm,
    lm_head
])

print(
    "Trainable parameters:",
    sum(p.numel() for p in model_parts.parameters())
)

Trainable parameters: 288288


## 24. Optimizer

I use AdamW to update the model's parameters based on the gradients
calculated during backpropagation.

The learning rate controls how large each parameter update is.

In [ ]:
optimizer = torch.optim.AdamW(
    model_parts.parameters(),
    lr=3e-4
)

print("Optimizer:", optimizer)

Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.01
)


## 25. Forward Pass

The previous cells demonstrated each Transformer component separately.

Now I combine those same operations into one function so the model can
repeat the complete process during training and generation.

This does not replace the earlier cells; those cells are kept to show how
each component works.

In [ ]:
def forward_pass(X):
    # 1. Token + position embeddings
    token_embeddings = token_embedding_table(X)

    T = X.size(1)

    positions = torch.arange(
        T,
        device=X.device
    )

    position_embeddings = position_embedding_table(positions)

    x = token_embeddings + position_embeddings

    # 2. Self-attention
    Q = query_layer(x)
    K = key_layer(x)
    V = value_layer(x)

    attention_scores = Q @ K.transpose(-2, -1)

    d_k = K.size(-1)

    scaled_scores = attention_scores / math.sqrt(d_k)

    # Dynamic causal mask
    causal_mask = torch.tril(
        torch.ones(
            T,
            T,
            device=X.device
        )
    )

    scaled_scores = scaled_scores.masked_fill(
        causal_mask == 0,
        float("-inf")
    )

    attention_weights = torch.softmax(
        scaled_scores,
        dim=-1
    )

    context = attention_weights @ V

    # 3. LayerNorm + Feed-Forward Network
    normalized_context = layer_norm(context)

    ffn_output = ffn(normalized_context)

    # 4. Residual connection
    residual_output = context + ffn_output

    # 5. Predict next character
    logits = lm_head(residual_output)

    return logits

In [ ]:
X, Y = get_batch("train")

logits = forward_pass(X)

print("Logits shape:", logits.shape)

Logits shape: torch.Size([32, 256, 288])


## 26. Train TinyGPT

During training, the model repeatedly:

1. Gets a batch of training sequences.
2. Predicts the next character.
3. Calculates the loss.
4. Computes gradients using backpropagation.
5. Updates the model parameters using AdamW.

I start with 500 training steps as an initial experiment.

In [ ]:
num_steps = 500

for step in range(num_steps):

    X, Y = get_batch("train")

    # Forward pass
    logits = forward_pass(X)

    # Calculate loss
    B, T, C = logits.shape

    logits_flat = logits.view(B * T, C)
    targets_flat = Y.view(B * T)

    loss = loss_fn(
        logits_flat,
        targets_flat
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(
            f"Step {step}: Loss = {loss.item():.4f}"
        )

Step 0: Loss = 5.6722
Step 100: Loss = 2.7700
Step 200: Loss = 2.6681
Step 300: Loss = 2.6199
Step 400: Loss = 2.6046


## 27. Validation

After training, I evaluate the model on a batch from the validation set.

The validation loss helps determine whether the model has learned useful
patterns beyond the exact sequences used during training.

In [ ]:
X_val, Y_val = get_batch("val")

logits_val = forward_pass(X_val)

B, T, C = logits_val.shape

logits_val_flat = logits_val.view(B * T, C)
targets_val_flat = Y_val.view(B * T)

val_loss = loss_fn(
    logits_val_flat,
    targets_val_flat
)

print("Validation loss:", val_loss.item())

Validation loss: 2.632357597351074


## 28. Generate Poetry

The trained model can now generate text autoregressively.

Starting from a prompt, it predicts one character, adds that character
to the context, predicts the next character, and repeats the process.

The model only uses the latest `block_size` characters as context.

In [ ]:
def generate_text(prompt, max_new_tokens=300):

    model_input = torch.tensor(
        [stoi[ch] for ch in prompt],
        dtype=torch.long
    ).unsqueeze(0)

    for _ in range(max_new_tokens):

        # Keep only the latest context
        context = model_input[:, -block_size:]

        # Predict next character
        logits = forward_pass(context)

        # Use only the final position
        logits = logits[:, -1, :]

        # Convert logits to probabilities
        probabilities = torch.softmax(
            logits,
            dim=-1
        )

        # Sample the next character
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Add the new character
        model_input = torch.cat(
            (model_input, next_token),
            dim=1
        )

    return "".join(
        itos[i.item()]
        for i in model_input[0]
    )

## 29. Test InkMind

I test the model using a natural-language poetry prompt.

The quality of the generated text will be used later when comparing
TinyGPT with GPT-2.

In [ ]:
prompt = "Love is a beautiful thing"

generated_text = generate_text(
    prompt,
    max_new_tokens=300
)

print(generated_text)

Love is a beautiful thinghon n. in swang caigrdit wen che   breny06oth is cortod ga  fwe sut

 ls lod f sme the  Da s ghesrtwon f ae Nigse Tre athe ifim  wDe lut sone our  A le w  cil  te thes ref sches we  an   m I llams imepe ourein t accheres  as . bie SFeeBaryome ted  in   cu d  farud wenye  cs  ea   we

Ae Tegre s  tin


In [ ]:
additional_steps = 500

for step in range(additional_steps):
    X, Y = get_batch("train")

    logits = forward_pass(X)

    B, T, C = logits.shape
    logits_flat = logits.view(B * T, C)
    targets_flat = Y.view(B * T)

    loss = loss_fn(logits_flat, targets_flat)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Additional step {step}: Loss = {loss.item():.4f}")

Additional step 0: Loss = 2.5477
Additional step 100: Loss = 2.5434
Additional step 200: Loss = 2.5433
Additional step 300: Loss = 2.4792
Additional step 400: Loss = 2.4231


In [ ]:
X_val, Y_val = get_batch("val")

logits_val = forward_pass(X_val)

B, T, C = logits_val.shape
logits_val_flat = logits_val.view(B * T, C)
targets_val_flat = Y_val.view(B * T)

val_loss = loss_fn(
    logits_val_flat,
    targets_val_flat
)

print(f"Validation Loss: {val_loss.item():.4f}")

Validation Loss: 2.4920


In [ ]:
prompt = "Love is a beautiful thing"

generated_text = generate_text(
    prompt,
    max_new_tokens=300
)

print(generated_text)

Love is a beautiful thingEw  warerd o ceathr adesh k s Cofoathas ousur, s re zandestele tOosighto.


Th,y e s any thin-an Au,  alou Pes,


Od chererk   at rowh ste
bee qucas ay isules he a acet?





Oun haAiqyonopispsssal Iyo, 


 ane phad Cofo 

amine


 d st  he se  “Lasth oveve.
 we Ie  omon  ftalogheanstt thosushooroto


## 30. Save the TinyGPT Model

I save the trained model parameters and optimizer state so that the model
can be restored after a Colab runtime restart.

The configuration and tokenizer mappings are also saved so the checkpoint
contains the information needed to reproduce the model.

In [ ]:
torch.save({
    "model_parts": model_parts.state_dict(),
    "optimizer": optimizer.state_dict(),
    "vocab_size": vocab_size,
    "n_embd": n_embd,
    "block_size": block_size,
    "stoi": stoi,
    "itos": itos
}, "inkmind_checkpoint_1000_steps.pth")

In [ ]:
import os

print(os.path.exists("inkmind_checkpoint_1000_steps.pth"))

True


In [ ]:
from google.colab import files

files.download("inkmind_checkpoint_1000_steps.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>